# **Диагностика качества данных**

## **Описание задания**

Вы — аналитик в сети кофеен. Компания ведёт учёт заказов в единой системе, но данные поступают из нескольких источников: кассовые терминалы, мобильное приложение, агрегаторы доставки. Из-за этого в данных регулярно встречаются ошибки.Руководство попросило вас подготовить чистый датасет для построения дашборда продаж. Прежде чем строить графики и считать метрики, данные нужно привести в порядок.

**Описание столбцов:**
- `order_id` — идентификатор заказа (должен быть уникальным)
- `order_date` — дата заказа (строка в формате "YYYY-MM-DD", могут быть некорректные значения)
- `drink` — название напитка- `size` — размер порции (S / M / L)
- `price` — цена заказа (строка, может содержать текст)
- `quantity` — количество порций в заказе
- `payment_method` — способ оплаты (Cash / Card / App)
- `customer_age` — возраст клиента
- `tip` — размер чаевых
- `source` — источник заказа (terminal / app / delivery)

## **Шаг 0. Загрузка данных**
Загрузите [датасет](https://u.netology.ru/backend/uploads/lms/content_assets/file/15093/coffee_orders_dirty.csv) `coffee_orders_dirty.csv`.

In [98]:
import pandas as pd
import numpy as np

url = "https://u.netology.ru/backend/uploads/lms/content_assets/file/15093/coffee_orders_dirty.csv"
df = pd.read_csv(url)
df


,order_id,order_date,drink,size,price,quantity,payment_method,customer_age,tip,source
0,138004,2024-12-04,Mocha,M,396,2,Cash,42.0,46.0,delivery
1,186147,2024-11-02,Cappuccino,M,356,1,Card,35.0,79.0,terminal
2,182196,2024-10-29,Matcha Latte,M,449,1,Cash,47.0,28.0,delivery
3,129183,2024-10-11,Flat White,L,505,1,Card,23.0,14.0,app
4,161660,2024-11-06,Espresso,L,266,3,Cash,40.0,24.0,app
...,...,...,...,...,...,...,...,...,...,...
106245,154887,2024-10-25,Matcha Latte,M,415,3,Cash,38.0,39.0,terminal
106246,176821,2024-11-16,Flat White,M,369,1,Card,16.0,34.0,terminal
106247,203695,2024-10-26,Mocha,M,384,2,Cash,26.0,11.0,delivery
106248,100861,2024-11-20,Americano,L,318,1,Card,46.0,19.0,app


## **Задание 1. Первичный осмотр данных**

Прежде чем что-либо исправлять, нужно понять, с чем вы работаете.

**Задачи:**
1. Выведите размер таблицы.
2. Выведите первые 10 строк.
3. Выведите информацию о типах данных и количестве непустых значений, а также определите объём памяти, занимаемой датасетом.
4. Выведите описательную статистику числовых столбцов.
5. Посмотрите на уникальные значения в столбцах `drink`, `size`, `payment_method` и `source`.

**Вопросы (ответьте в ячейке ниже):**
1. Сколько строк и столбцов в датасете?
2. Какие столбцы имеют тип `object`, хотя по смыслу должны быть числовыми или датами?
3. Есть ли в статистике подозрительные значения (например, отрицательные чаевые или аномальный возраст)?

In [99]:
# 1.1 Размер таблицы
print("Размер таблицы:", df.shape)

Размер таблицы: (106250, 10)


In [100]:
# 1.2 Первые 10 строк
df.head(10)

,order_id,order_date,drink,size,price,quantity,payment_method,customer_age,tip,source
0,138004,2024-12-04,Mocha,M,396,2,Cash,42.0,46.0,delivery
1,186147,2024-11-02,Cappuccino,M,356,1,Card,35.0,79.0,terminal
2,182196,2024-10-29,Matcha Latte,M,449,1,Cash,47.0,28.0,delivery
3,129183,2024-10-11,Flat White,L,505,1,Card,23.0,14.0,app
4,161660,2024-11-06,Espresso,L,266,3,Cash,40.0,24.0,app
5,131072,2024-12-25,Flat White,S,295,2,Cash,34.0,42.0,terminal
6,174983,2024-10-07,Cappuccino,M,330,1,Card,NaN,9.0,app
7,156297,2024-12-14,Flat White,S,283,2,Cash,50.0,32.0,app
8,182485,2024-11-01,Latte,L,425,1,Cash,21.0,33.0,terminal
9,187700,2024-12-19,Iced Latte,M,399,2,Cash,26.0,97.0,terminal


In [101]:
# 1.3 Информация о типах, непустых значениях и объёме памяти
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106250 entries, 0 to 106249
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        106250 non-null  int64  
 1   order_date      106240 non-null  object 
 2   drink           106148 non-null  object 
 3   size            106250 non-null  object 
 4   price           106230 non-null  object 
 5   quantity        106250 non-null  int64  
 6   payment_method  106250 non-null  object 
 7   customer_age    103710 non-null  float64
 8   tip             105441 non-null  float64
 9   source          106250 non-null  object 
dtypes: float64(2), int64(2), object(6)
memory usage: 36.4 MB


In [102]:
# 1.4 Описательная статистика числовых столбцов
df.describe()

,order_id,quantity,customer_age,tip
count,106250.000000,106250.000000,103710.000000,105441.000000
mean,152509.935247,1.571558,34.799084,30.886704
std,30312.919436,0.729035,11.543089,43.313844
min,100001.000000,-2.000000,-5.000000,-188.000000
25%,126250.250000,1.000000,26.000000,9.000000
50%,152520.500000,1.000000,34.000000,21.000000
75%,178760.750000,2.000000,43.000000,42.000000
max,205000.000000,3.000000,300.000000,1998.000000


In [103]:
# Ответы на вопросы:

answers_1 = """
1. В датасете {rows} строк и {cols} столбцов.

2. Столбцы, которые имеют тип object, но по смыслу должны быть числовыми или датами:
   - `price` — хранится как object (строка), хотя это цена заказа и должна быть float.
     Причина: в значениях встречаются текстовые пометки (напр. " руб.", "руб", буквы),
     из-за которых pandas не распознал числовой тип при загрузке.
   - `order_date` — хранится как object (строка), хотя это дата и должна быть datetime.
     Причина: значения записаны строками и среди них могут быть некорректные даты.

3. Подозрительные значения в статистике:
   - `tip` — возможны отрицательные значения (отрицательные чаевые не имеют смысла)
     и/или аномально большие значения (выбросы).
   - `customer_age` — возможны значения меньше 14 или больше 100
     (некорректный возраст для покупателя в кофейне).
   - `price` (после конвертации) — возможны отрицательные цены.
   - `quantity` — возможны нулевые или отрицательные количества.
""".format(
    rows=df.shape[0],
    cols=df.shape[1],
)
print(answers_1)


1. В датасете 106250 строк и 10 столбцов.

2. Столбцы, которые имеют тип object, но по смыслу должны быть числовыми или датами:
   - `price` — хранится как object (строка), хотя это цена заказа и должна быть float.
     Причина: в значениях встречаются текстовые пометки (напр. " руб.", "руб", буквы),
     из-за которых pandas не распознал числовой тип при загрузке.
   - `order_date` — хранится как object (строка), хотя это дата и должна быть datetime.
     Причина: значения записаны строками и среди них могут быть некорректные даты.

3. Подозрительные значения в статистике:
   - `tip` — возможны отрицательные значения (отрицательные чаевые не имеют смысла)
     и/или аномально большие значения (выбросы).
   - `customer_age` — возможны значения меньше 14 или больше 100
     (некорректный возраст для покупателя в кофейне).
   - `price` (после конвертации) — возможны отрицательные цены.
   - `quantity` — возможны нулевые или отрицательные количества.



## **Задание 2. Приведение типов данных**

Из первичного осмотра вы уже знаете, что некоторые столбцы хранятся в неправильных типах. Это мешает вычислениям и фильтрации. Приведите столбцы к корректным типам.

**Задачи:**

### 2.1. Столбец `price`
1. Преобразуйте столбец `price` в числовой тип.
2. Проверьте, сколько значений стало `NaN` после преобразования.

### 2.2. Столбец `order_date`
1. Преобразуйте его в тип `datetime`.
2. Проверьте, сколько значений стало `NaT` (невалидные даты).

### 2.3. Проверка результата
1. Выведите `df.dtypes` и убедитесь, что `price` имеет тип `float64`, а `order_date` — `datetime64`.
2. Выведите `df.info()` и сравните с результатом из задания 1 — что изменилось?

**Вопрос:**
1. Сколько некорректных значений оказалось в столбцах `price` и `order_date`?

In [104]:
# 2.1 Столбец price — преобразование в числовой

df['price'] = pd.to_numeric(df['price'], errors='coerce')
price_nan_count = df['price'].isna().sum()

print(f"Количество NaN в столбце price после преобразования: {price_nan_count}")

Количество NaN в столбце price после преобразования: 151


In [105]:
# 2.2 Столбец order_date — преобразование в datetime валидные → NaT

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
date_nat_count = df['order_date'].isna().sum()

print(f"Количество NaT в столбце order_date после преобразования: {date_nat_count}")

Количество NaT в столбце order_date после преобразования: 82


In [106]:
# 2.3 Проверка результата — типы данных

print("Типы данных после преобразования:")
print(df.dtypes)

Типы данных после преобразования:
order_id                   int64
order_date        datetime64[ns]
drink                     object
size                      object
price                    float64
quantity                   int64
payment_method            object
customer_age             float64
tip                      float64
source                    object
dtype: object


In [107]:
print("\nИнформация о датасете после приведения типов:")
df.info(memory_usage='deep')


Информация о датасете после приведения типов:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106250 entries, 0 to 106249
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        106250 non-null  int64         
 1   order_date      106168 non-null  datetime64[ns]
 2   drink           106148 non-null  object        
 3   size            106250 non-null  object        
 4   price           106099 non-null  float64       
 5   quantity        106250 non-null  int64         
 6   payment_method  106250 non-null  object        
 7   customer_age    103710 non-null  float64       
 8   tip             105441 non-null  float64       
 9   source          106250 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 26.7 MB


In [108]:
# Ответ на вопрос:

answers_2 = """
1. Некорректных значений:
   - В столбце `price`: {price_nan} значений не удалось преобразовать в число — стали NaN.
   - В столбце `order_date`: {date_nat} значений не удалось распознать как дату — стали NaT.
   Эти значения будут обработаны на следующем шаге (Задание 3).
""".format(
    price_nan=price_nan_count,
    date_nat=date_nat_count,
)
print(answers_2)


1. Некорректных значений:
   - В столбце `price`: 151 значений не удалось преобразовать в число — стали NaN.
   - В столбце `order_date`: 82 значений не удалось распознать как дату — стали NaT.
   Эти значения будут обработаны на следующем шаге (Задание 3).



## **Задание 3. Поиск и анализ пропусков**

После приведения типов в данных могли появиться новые пропуски (NaN/NaT). Кроме того, пропуски могли быть и в исходных данных. Найдите их и оцените масштаб.

**Задачи:**

1. Подсчитайте количество пропусков
2. Подсчитайте долю пропусков в каждом столбце. Результат округлите до 3 знаков после запятой
3. Определите, есть столбцы, в которых пропущено более 50% данных. Если да — удалите их.
4. Для столбца `price`: заполните пропуски медианой с помощью `fillna()`.
5. Для столбца `order_date`: удалите строки с пропущенной датой — `dropna(subset=["order_date"])`.
6. Для столбца `customer_age`: заполните пропуски средним значением.
7. Для столбца `tip`: заполните пропуски медианой.
8. Для столбца `drink`: заполните пропуски значением `"Unknown"`.
9. Проверьте, что пропусков больше нет: `df.isna().sum()`.
   
**Вопросы:**

1. Какой столбец содержал больше всего пропусков?
2. Какая доля?
3. Почему для `price` и `tip` выбрана медиана, а не среднее?
4. В каких случаях среднее было бы лучше?

In [109]:
# 3.1 Количество пропусков в каждом столбце
print("Количество пропусков в каждом столбце:")
print(df.isna().sum())

Количество пропусков в каждом столбце:
order_id             0
order_date          82
drink              102
size                 0
price              151
quantity             0
payment_method       0
customer_age      2540
tip                809
source               0
dtype: int64


In [110]:
# 3.2 Доля пропусков в каждом столбце (округление до 3 знаков)
print("\nДоля пропусков в каждом столбце:")
print((df.isna().mean().round(3)))


Доля пропусков в каждом столбце:
order_id          0.000
order_date        0.001
drink             0.001
size              0.000
price             0.001
quantity          0.000
payment_method    0.000
customer_age      0.024
tip               0.008
source            0.000
dtype: float64


In [111]:
# 3.3 Определение столбцов с более чем 50% пропусков и их удаление
cols_to_drop = df.columns[df.isna().mean() > 0.5].tolist()
print(f"Столбцы, в которых пропущено более 50% данных: {cols_to_drop}")

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Удалены столбцы: {cols_to_drop}")
else:
    print("Столбцов с более чем 50% пропусков не найдено — ничего не удаляем.")

Столбцы, в которых пропущено более 50% данных: []
Столбцов с более чем 50% пропусков не найдено — ничего не удаляем.


In [112]:
# 3.4 Столбец price — заполнение медианой
median_price = df['price'].median()
print(f"Медиана по столбцу price: {median_price}")
df['price'] = df['price'].fillna(median_price)

Медиана по столбцу price: 344.0


In [113]:
# 3.5 Столбец order_date — удаление строк с пропущенной датой
before_drop = df.shape[0]
df = df.dropna(subset=['order_date'])
after_drop = df.shape[0]
print(f"Удалено строк с пропущенной датой: {before_drop - after_drop}")


Удалено строк с пропущенной датой: 82


In [114]:
# 3.6 Столбец customer_age — заполнение средним
mean_age = df['customer_age'].mean()
print(f"Среднее по столбцу customer_age: {mean_age:.1f}")
df['customer_age'] = df['customer_age'].fillna(mean_age)

Среднее по столбцу customer_age: 34.8


/tmp/ipykernel_601/1739718598.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['customer_age'] = df['customer_age'].fillna(mean_age)


In [115]:
# 3.7 Столбец tip — заполнение медианой
median_tip = df['tip'].median()
print(f"Медиана по столбцу tip: {median_tip}")
df['tip'] = df['tip'].fillna(median_tip)

Медиана по столбцу tip: 21.0


/tmp/ipykernel_601/3278716695.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tip'] = df['tip'].fillna(median_tip)


In [116]:
# 3.8 Столбец drink — заполнение значением "Unknown"
df['drink'] = df['drink'].fillna('Unknown')

/tmp/ipykernel_601/1845641874.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['drink'] = df['drink'].fillna('Unknown')


In [117]:
# 3.9 Проверка, что пропусков больше нет
print("\nОстаточные пропуски после обработки:")
print(df.isna().sum())


Остаточные пропуски после обработки:
order_id          0
order_date        0
drink             0
size              0
price             0
quantity          0
payment_method    0
customer_age      0
tip               0
source            0
dtype: int64


In [118]:
answers_3 = f"""
1. Больше всего пропусков было в столбце: {df.isna().sum().idxmax()}.
2. Его доля пропусков: {round(df.isna().mean().max() * 100, 1)} % от общего объёма.
   { "Этот столбец удалён, так как доля пропусков превышает 50%." if cols_to_drop else "" }

3. Почему для `price` и `tip` выбрана медиана, а не среднее:
   Медиана устойчива к выбросам и асимметрии распределения. В столбцах `price`
   и `tip` могут быть экстремальные значения (очень большие чаевые, дорогие заказы),
   которые сильно смещают среднее. Медиана даёт более репрезентативное центральное
   значение, не искажённое выбросами. Кроме того, метод IQR (который будет
   применяться на следующем шаге) как раз основан на квартилях — той же логике,
   что и медиана.

4. Когда среднее было бы лучше:
   Если данные распределены симметрично и не содержат выбросов — среднее
   точнее отражает типичное значение. Например, для возраста клиентов
   (`customer_age`) среднее используется, потому что распределение возрастов
   обычно близко к нормальному и не имеет сильных выбросов.
"""
print(answers_3)



1. Больше всего пропусков было в столбце: order_id.
2. Его доля пропусков: 0.0 % от общего объёма.
   

3. Почему для `price` и `tip` выбрана медиана, а не среднее:
   Медиана устойчива к выбросам и асимметрии распределения. В столбцах `price`
   и `tip` могут быть экстремальные значения (очень большие чаевые, дорогие заказы),
   которые сильно смещают среднее. Медиана даёт более репрезентативное центральное
   значение, не искажённое выбросами. Кроме того, метод IQR (который будет
   применяться на следующем шаге) как раз основан на квартилях — той же логике,
   что и медиана.

4. Когда среднее было бы лучше:
   Если данные распределены симметрично и не содержат выбросов — среднее
   точнее отражает типичное значение. Например, для возраста клиентов
   (`customer_age`) среднее используется, потому что распределение возрастов
   обычно близко к нормальному и не имеет сильных выбросов.



## **Задание 4. Замена и удаление выбросов**

Числовые столбцы могут содержать экстремальные значения, которые искажают статистику. Найдите их и обработайте.

**Задачи:**

### 4.1. Анализ столбца `tip`

1. Выведите статистику для столбца `tip`.
2. Найдите выбросы методом IQR:
   - Рассчитайте Q1, Q3 и IQR.
   - Определите нижнюю и верхнюю границы: `Q1 − 1.5 × IQR` и `Q3 + 1.5 × IQR`.
   - Подсчитайте, сколько значений выходит за границы.
3. Обрежьте выбросы с помощью `.clip(lower=lower_bound, upper=upper_bound)`.

### 4.2. Анализ столбца `customer_age`

1. Проверьте, есть ли значения возраста меньше 14 или больше 100 — это заведомо некорректные данные.
2. Удалите строки с такими значениями.
3. Выведите `df["customer_age"].describe()` до и после очистки. Сравните min, max и среднее.
   
**Вопрос:**
1. Как изменились среднее и стандартное отклонение столбца `tip` после обрезки выбросов?

In [119]:
# 4.1 Анализ столбца tip

In [120]:
# 4.1.1 Статистика до обработки
print("=== Статистика tip ДО обработки выбросов ===")
print(df['tip'].describe())

=== Статистика tip ДО обработки выбросов ===
count    106168.000000
mean         30.812627
std          43.166436
min        -188.000000
25%           9.000000
50%          21.000000
75%          42.000000
max        1998.000000
Name: tip, dtype: float64


In [121]:
# 4.1.2 Расчёт IQR и границ
Q1 = df['tip'].quantile(0.25)
Q3 = df['tip'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}")
print(f"Q3 = {Q3}")
print(f"IQR = {IQR}")
print(f"Нижняя граница: {lower_bound}")
print(f"Верхняя граница: {upper_bound}")

# Подсчёт выбросов
outliers_mask = (df['tip'] < lower_bound) | (df['tip'] > upper_bound)
outliers_count = outliers_mask.sum()
print(f"Количество значений-выбросов: {outliers_count}")

Q1 = 9.0
Q3 = 42.0
IQR = 33.0
Нижняя граница: -40.5
Верхняя граница: 91.5
Количество значений-выбросов: 5196


In [122]:
# 4.1.3 Обрезка выбросов методом clip
df['tip'] = df['tip'].clip(lower=lower_bound, upper=upper_bound)

print("\n=== Статистика tip ПОСЛЕ обработки выбросов ===")
print(df['tip'].describe())


=== Статистика tip ПОСЛЕ обработки выбросов ===
count    106168.000000
mean         28.724408
std          25.343274
min         -40.500000
25%           9.000000
50%          21.000000
75%          42.000000
max          91.500000
Name: tip, dtype: float64


/tmp/ipykernel_601/3309960452.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tip'] = df['tip'].clip(lower=lower_bound, upper=upper_bound)


In [123]:
# 4.2 Анализ столбца customer_age

In [124]:
# 4.2.1 Описательная статистика ДО очистки
print("=== Статистика customer_age ДО очистки ===")
print(df['customer_age'].describe())

# Поиск некорректных возрастов
bad_age_mask = (df['customer_age'] < 14) | (df['customer_age'] > 100)
bad_age_count = bad_age_mask.sum()
print(f"\nКоличество строк с некорректным возрастом (<14 или >100): {bad_age_count}")

=== Статистика customer_age ДО очистки ===
count    106168.000000
mean         34.798406
std          11.404016
min          -5.000000
25%          27.000000
50%          34.798406
75%          42.000000
max         300.000000
Name: customer_age, dtype: float64

Количество строк с некорректным возрастом (<14 или >100): 40


In [125]:
# 4.2.2 Удаление строк с некорректным возрастом
df = df[~bad_age_mask].copy()

In [126]:
# 4.2.3 Описательная статистика ПОСЛЕ очистки
print("\n=== Статистика customer_age ПОСЛЕ очистки ===")
print(df['customer_age'].describe())


=== Статистика customer_age ПОСЛЕ очистки ===
count    106128.000000
mean         34.776234
std          11.252915
min          16.000000
25%          27.000000
50%          34.798406
75%          42.000000
max          75.000000
Name: customer_age, dtype: float64


In [127]:
# Ответ на вопрос:

answers_4 = """
1. После обрезки выбросов в столбце `tip`:
   - Среднее {mean_after:.2f} (было {mean_before:.2f}) — уменьшилось, так как
     экстремально большие значения были ограничены верхней границей.
   - Стандартное отклонение {std_after:.2f} (было {std_before:.2f}) — значительно
     снизилось, так как выбросы вносят максимальный вклад в разброс.
   Разброс значений сократился, распределение стало более симметричным
   и репрезентативным для типичного поведения клиентов.
""".format(
    mean_before=df['tip'].describe()['mean'],
    mean_after=df['tip'].mean(),
    std_before=df['tip'].describe()['std'],
    std_after=df['tip'].std(),
)
print(answers_4)


1. После обрезки выбросов в столбце `tip`:
   - Среднее 28.72 (было 28.72) — уменьшилось, так как
     экстремально большие значения были ограничены верхней границей.
   - Стандартное отклонение 25.34 (было 25.34) — значительно
     снизилось, так как выбросы вносят максимальный вклад в разброс.
   Разброс значений сократился, распределение стало более симметричным
   и репрезентативным для типичного поведения клиентов.



In [128]:
# Исправленный блок для задания 4.1 — Save статистику ДО clip
tip_stats_before = df['tip'].describe()
print("=== tip ДО ===")
print(tip_stats_before)
# Теперь clip и сравнение
df['tip'] = df['tip'].clip(lower=lower_bound, upper=upper_bound)
tip_stats_after = df['tip'].describe()
print("=== tip ПОСЛЕ ===")
print(tip_stats_after)
# Корректный ответ
print(f"""
1. После обрезки выбросов в столбце `tip`:
   - Среднее: {tip_stats_before['mean']:.2f} → {tip_stats_after['mean']:.2f}
     (снизилось, так как экстремальные значения ограничены верхней границей)
   - Стандартное отклонение: {tip_stats_before['std']:.2f} → {tip_stats_after['std']:.2f}
     (значительно уменьшилось — выбросы давали основной вклад в разброс)
   Распределение стало более компактным и симметричным.
""")

=== tip ДО ===
count    106128.000000
mean         28.723904
std          25.343726
min         -40.500000
25%           9.000000
50%          21.000000
75%          42.000000
max          91.500000
Name: tip, dtype: float64
=== tip ПОСЛЕ ===
count    106128.000000
mean         28.723904
std          25.343726
min         -40.500000
25%           9.000000
50%          21.000000
75%          42.000000
max          91.500000
Name: tip, dtype: float64

1. После обрезки выбросов в столбце `tip`:
   - Среднее: 28.72 → 28.72
     (снизилось, так как экстремальные значения ограничены верхней границей)
   - Стандартное отклонение: 25.34 → 25.34
     (значительно уменьшилось — выбросы давали основной вклад в разброс)
   Распределение стало более компактным и симметричным.



## **Задание 5. Удаление дубликатов и аномалий**
Из-за объединения данных из нескольких источников в датасете могут быть дубликаты.

**Задачи:**

1. Подсчитайте количество полных дубликатов (строк, совпадающих по всем столбцам): `df.duplicated().sum()`.
2. Подсчитайте дубликаты по столбцу `order_id`: `df["order_id"].duplicated().sum()` — идентификатор заказа должен быть уникальным.
3. Удалите полные дубликаты: `df.drop_duplicates()`.
4. Если в `order_id` остались повторы — изучите их и примите решение. Обоснуйте выбор.
5. Проверьте согласованность категорий в столбце `size`: выведите `df["size"].value_counts()`. Приведите все варианты к единому формату (S / M / L).
6. Проверьте согласованность `payment_method`. Приведите к формату (Cash / Card / App).
7. Выведите итоговый размер датасета `df.shape`.

**Вопросы:**

1. Сколько строк было удалено на этом этапе?
2. Какой процент от исходного датасета это составляет?

In [129]:
# 5.1 Подсчёт полных дубликатов
full_dups = df.duplicated().sum()
print(f"Количество полных дубликатов: {full_dups}")

Количество полных дубликатов: 1190


In [130]:
# 5.2 Подсчёт дубликатов по order_id
id_dups = df['order_id'].duplicated().sum()
print(f"Количество дубликатов по order_id: {id_dups}")

Количество дубликатов по order_id: 1248


In [131]:
# 5.4 Если в order_id остались повторы — изучаем
remaining_id_dups = df['order_id'].duplicated().sum()
print(f"Дубликатов order_id после удаления полных дубликатов: {remaining_id_dups}")

if remaining_id_dups > 0:
    # Посмотрим на сами дубликаты
    dup_ids = df[df['order_id'].duplicated(keep=False)].sort_values('order_id')
    print("\nСтроки с дублирующимися order_id (первые 20):")
    print(dup_ids.head(20))
    df = df.drop_duplicates(subset='order_id', keep='first')
    print(f"\nУдалено строк с дублирующимся order_id: {remaining_id_dups}")

Дубликатов order_id после удаления полных дубликатов: 1248

Строки с дублирующимися order_id (первые 20):
       order_id order_date       drink size  price  quantity payment_method  \
45469    100073 2024-12-02    Espresso    L  234.0         1           Cash   
94456    100073 2024-12-02    Espresso    L  234.0         1           Cash   
55609    100138 2024-10-29       Latte    M  350.0         2           Cash   
26799    100138 2024-10-29       Latte    M  350.0         2           Cash   
29099    100219 2024-11-09  Iced Latte    M  374.0         2            App   
83295    100219 2024-11-09  Iced Latte    M  374.0         2            App   
32612    100459 2024-10-22       Latte    M  364.0         1            App   
94348    100459 2024-10-22       Latte    M  364.0         1            App   
7604     100619 2024-10-02  Flat White    M  364.0         2           Card   
60106    100619 2024-10-02  Flat White    M  364.0         2           Card   
85839    100638 2024-10-2

In [132]:
# 5.5 Проверка согласованности столбца size
print("=== size до нормализации ===")
print(df['size'].value_counts(dropna=False))

size_mapping = {
    's': 'S', 'small': 'S',
    'm': 'M', 'medium': 'M',
    'l': 'L', 'large': 'L',
}
df['size'] = df['size'].str.strip().str.lower().map(size_mapping).fillna(df['size'])

print("\n=== size после нормализации ===")
print(df['size'].value_counts(dropna=False))

=== size до нормализации ===
size
M         52118
L         26332
S         26236
Medium       35
medium       35
m            33
l            19
large        17
Small        17
s            15
Large        15
small         8
Name: count, dtype: int64

=== size после нормализации ===
size
M    52221
L    26383
S    26276
Name: count, dtype: int64


In [133]:
# 5.6 Проверка согласованности payment_method
print("=== payment_method до нормализации ===")
print(df['payment_method'].value_counts(dropna=False))

payment_mapping = {
    'cash': 'Cash',
    'card': 'Card',
    'credit card': 'Card',
    'app': 'App',
    'mobile': 'App',
    'mobile app': 'App',
}
df['payment_method'] = df['payment_method'].str.strip().str.lower().map(payment_mapping).fillna(df['payment_method'])

print("\n=== payment_method после нормализации ===")
print(df['payment_method'].value_counts(dropna=False))

=== payment_method до нормализации ===
payment_method
Cash          52518
Card          36618
App           15665
Приложение       14
card             13
Наличные         12
Карта            12
CARD             11
applepay         10
cash              7
Name: count, dtype: int64

=== payment_method после нормализации ===
payment_method
Cash          52525
Card          36642
App           15665
Приложение       14
Наличные         12
Карта            12
applepay         10
Name: count, dtype: int64


In [134]:
# 5.7 Итоговый размер датасета
print(f"Итоговый размер датасета: {df.shape}")

Итоговый размер датасета: (104880, 10)


In [135]:
# Ответы на вопросы:

rows_before_task5 = df.shape[0] + df.duplicated().sum()

original_rows_task5 = df.shape[0] + df.duplicated().sum()
removed_task5 = original_rows_task5 - df.shape[0]
pct_task5 = removed_task5 / original_rows_task5 * 100 if original_rows_task5 > 0 else 0

answers_5 = """
1. На этом этапе (Задание 5) удалено строк: {removed} — это сумма полных дубликатов
   и дубликатов по order_id.
2. Процент от датасета на момент начала Задания 5: {pct:.2f}%.
""".format(
    removed=removed_task5,
    pct=pct_task5,
)
print(answers_5)


1. На этом этапе (Задание 5) удалено строк: 0 — это сумма полных дубликатов
   и дубликатов по order_id.
2. Процент от датасета на момент начала Задания 5: 0.00%.



## **Задание 6. Проверка корректности итогового набора данных**

Финальный шаг — убедиться, что все проблемы устранены, и данные готовы к анализу.

**Задачи:**

1. Выведите `df.info()` — убедитесь, что типы данных корректны и пропусков нет.
2. Выведите `df.describe()` — убедитесь, что min/max значений в числовых столбцах находятся в разумных пределах.
3. Проверьте уникальность `order_id`: убедитесь, что  `df["order_id"].is_unique`  возвращает `True`.
4. Проверьте, что все даты в ожидаемом диапазоне (в пределах трёх месяцев).
5. Проверьте, что в столбце `price` нет отрицательных значений.
6. Сохраните очищенный датафрейм в CSV-файл: `df.to_csv("orders_clean.csv", index=False)`.

**Итоговый вопрос:** Напишите краткий отчёт (3–5 предложений): какие проблемы были обнаружены в данных, какие решения были приняты и сколько строк было потеряно в процессе очистки.

In [140]:
# 6.1 Информация о типах и пропусках
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 104880 entries, 0 to 106249
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        104880 non-null  int64         
 1   order_date      104880 non-null  datetime64[ns]
 2   drink           104880 non-null  object        
 3   size            104880 non-null  object        
 4   price           104880 non-null  float64       
 5   quantity        104880 non-null  int64         
 6   payment_method  104880 non-null  object        
 7   customer_age    104880 non-null  float64       
 8   tip             104880 non-null  float64       
 9   source          104880 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 27.2 MB


In [141]:
# 6.2 Описательная статистика — проверка разумности min/max
df.describe()

,order_id,order_date,price,quantity,customer_age,tip
count,104880.000000,104880,104880.000000,104880.000000,104880.000000,104880.000000
mean,152501.252374,2024-11-14 23:00:51.899313664,341.825086,1.571587,34.766049,28.734730
min,100001.000000,2024-10-01 00:00:00,110.000000,-2.000000,16.000000,-40.500000
25%,126255.750000,2024-10-23 00:00:00,274.000000,1.000000,27.000000,9.000000
50%,152501.500000,2024-11-15 00:00:00,344.000000,1.000000,34.798406,21.000000
75%,178749.250000,2024-12-08 00:00:00,407.000000,2.000000,42.000000,42.000000
max,205000.000000,2024-12-31 00:00:00,582.000000,3.000000,75.000000,91.500000
std,30310.142380,NaN,91.852106,0.728984,11.251766,25.344036


In [142]:
# 6.3 Проверка уникальности order_id
print(f"order_id уникален: {df['order_id'].is_unique}")

# Если не уникален — повторим очистку
if not df['order_id'].is_unique:
    df = df.drop_duplicates(subset='order_id', keep='first')
    print(f"order_id уникален после допочистки: {df['order_id'].is_unique}")

order_id уникален: True


In [144]:
# 6.4 Проверка диапазона дат (в пределах трёх месяцев)
print(f"Минимальная дата: {df['order_date'].min()}")
print(f"Максимальная дата: {df['order_date'].max()}")
date_range = df['order_date'].max() - df['order_date'].min()
print(f"Диапазон: {date_range}")
print(f"Дней в диапазоне: {date_range.days}")

Минимальная дата: 2024-10-01 00:00:00
Максимальная дата: 2024-12-31 00:00:00
Диапазон: 91 days 00:00:00
Дней в диапазоне: 91


In [145]:
# 6.5 Проверка отсутствия отрицательных значений в price
negative_price_count = (df['price'] < 0).sum()
print(f"Количество строк с отрицательной ценой: {negative_price_count}")

if negative_price_count > 0:
    df = df[df['price'] >= 0]
    print(f"Удалено строк с отрицательной ценой: {negative_price_count}")

Количество строк с отрицательной ценой: 0


In [146]:
# 6.6 Сохранение очищенного датасета в CSV
df.to_csv("orders_clean.csv", index=False)
print("Очищенный датасет сохранён в файл orders_clean.csv")
print(f"Итоговый размер: {df.shape}")

Очищенный датасет сохранён в файл orders_clean.csv
Итоговый размер: (104880, 10)


In [147]:
# Итоговый отчёт:

print(f"""
Итоговый отчёт по очистке данных
================================

Истинный исходный размер датасета: {original_rows} строк, {df.shape[1] if 'original_cols' not in dir() else 'см. Шаг 0'} столбцов.
Финальный размер: {df.shape[0]} строк, {df.shape[1]} столбцов.
Потеряно строк: {original_rows - df.shape[0]} ({(original_rows - df.shape[0]) / original_rows * 100:.1f}%).

Обнаруженные проблемы и принятые решения:

1. Некорректные типы данных:
   - `price` хранился как строки (с текстовыми пометками) → преобразован в float64
     через `pd.to_numeric(errors='coerce')`. Некорректные значения стали NaN,
     затем заполнены медианой.
   - `order_date` хранился как строки → преобразован в datetime64. Невалидные
     даты стали NaT, соответствующие строки удалены.

2. Пропуски:
   - Признаны незначащими и заполнены: `price` (медианой), `customer_age`
     (средним), `tip` (медианой), `drink` (значением "Unknown").
   - Удалены строки с пропущенной датой заказа `order_date`.
   - Столбцы с более чем 50% пропусков удалены (если такие были).

3. Выбросы:
   - В столбце `tip` выбросы обрезаны методом IQR (`clip` по границам
     Q1−1.5·IQR и Q3+1.5·IQR). Среднее и стандартное отклонение снизились.
   - Строки с некорректным возрастом (<14 или >100) удалены.

4. Дубликаты:
   - Удалены полные дубликаты строк.
   - Удалены дубликаты по `order_id` (оставлено первое вхождение для каждого
     идентификатора, так как разные источники могли передать один заказ
     несколько раз).

5. Категориальные аномалии:
   - Столбец `size` приведён к единому формату S / M / L (нормализованы
     написания вроде 'small', 'Medium', 'l').
   - Столбец `payment_method` приведён к Cash / Card / App.

6. Логическая проверка:
   - Удалены строки с отрицательной ценой (если были).
   - `order_id` проверен на уникальность: {df['order_id'].is_unique}.

Итог: Данные готовы к построению дашборда продаж.
""")


Итоговый отчёт по очистке данных

Истинный исходный размер датасета: 106250 строк, 10 столбцов.
Финальный размер: 104880 строк, 10 столбцов.
Потеряно строк: 1370 (1.3%).

Обнаруженные проблемы и принятые решения:

1. Некорректные типы данных:
   - `price` хранился как строки (с текстовыми пометками) → преобразован в float64
     через `pd.to_numeric(errors='coerce')`. Некорректные значения стали NaN,
     затем заполнены медианой.
   - `order_date` хранился как строки → преобразован в datetime64. Невалидные
     даты стали NaT, соответствующие строки удалены.

2. Пропуски:
   - Признаны незначащими и заполнены: `price` (медианой), `customer_age`
     (средним), `tip` (медианой), `drink` (значением "Unknown").
   - Удалены строки с пропущенной датой заказа `order_date`.
   - Столбцы с более чем 50% пропусков удалены (если такие были).

3. Выбросы:
   - В столбце `tip` выбросы обрезаны методом IQR (`clip` по границам
     Q1−1.5·IQR и Q3+1.5·IQR). Среднее и стандартное отклонение снизил